# 🚀 Sistema de Control de Vuelo de Cohete

**Autor:** Omar Velázquez (https://www.linkedin.com/in/ovelazquezj)

**Licencia:** CC BY 4.0 (2025)

---

Este notebook simula el sistema de control de vuelo de un cohete desde el despegue hasta la órbita. Implementa conceptos avanzados como:

- **Tabla hash con plan de vuelo** usando pandas
- **AST (Árbol de Sintaxis Abstracta)** para control sin `if` explícitos
- **Filtro de Kalman** para suavizar mediciones ruidosas
- **Controlador PID** para corrección de desviaciones

Simularemos dos escenarios: vuelo nominal y vuelo con viento.

## 📦 Instalación de dependencias

**¿Qué estamos haciendo aquí?**

Vamos a instalar Graphviz, una herramienta que nos permite dibujar árboles y grafos. Piensa en ella como un programa que convierte descripciones de texto en diagramas bonitos. La necesitamos para visualizar el AST del sistema de control.

In [ ]:
!apt-get install -y graphviz
!pip install graphviz -q

## 📚 Importación de bibliotecas

**¿Qué son estas bibliotecas?**

- **numpy**: Es como una calculadora superpotente para hacer operaciones matemáticas con muchos números a la vez.
- **pandas**: Nos ayuda a trabajar con tablas de datos, como si fuera Excel pero en código.
- **matplotlib**: Es nuestro pincel para dibujar gráficas y visualizar datos.
- **hashlib**: Nos permite crear "huellas digitales" únicas (hash) de los datos.
- **graphviz**: La herramienta para dibujar el árbol de decisiones del cohete.
- **time**: Para simular el paso del tiempo en la simulación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hashlib
import graphviz
import time
from IPython.display import display

## 🗺️ Plan de Vuelo con Tabla Hash

**¿Qué es una tabla hash?**

Imagina que tienes un libro gigante con instrucciones para el cohete. En lugar de leer página por página, usas un índice mágico (el hash) que te lleva directamente a la página correcta. El hash es como una huella digital única de cada altitud.

**¿Por qué usar SHA1?**

SHA1 es un algoritmo que convierte cualquier dato en una cadena única de caracteres. Es como convertir "1000 metros" en una contraseña única que identifica esa altitud.

En nuestro plan de vuelo:
- **Altitud**: Altura del cohete
- **Fase**: Etapa del vuelo (despegue, ascenso, órbita)
- **Acción**: Qué debe hacer el cohete
- **hash_id**: Huella digital única para buscar rápido

In [ ]:
def crear_plan_vuelo():
    """Crea el plan de vuelo como una tabla hash"""
    
    altitudes = [0, 1000, 5000, 10000, 20000, 50000, 100000, 150000, 200000]
    fases = ['despegue', 'ascenso_inicial', 'ascenso', 'ascenso_medio', 
             'ascenso_alto', 'pre_orbital', 'orbital', 'orbital_medio', 'orbital_final']
    acciones = ['encender_motores', 'mantener_empuje', 'ajustar_vectores', 
                'control_activo', 'reducir_empuje', 'preparar_orbital', 
                'insercion_orbital', 'estabilizar', 'orbita_establecida']
    
    plan = pd.DataFrame({
        'altitud': altitudes,
        'fase': fases,
        'accion': acciones
    })
    
    # Generar hash_id usando SHA1 de la altitud
    plan['hash_id'] = plan['altitud'].apply(
        lambda x: hashlib.sha1(str(x).encode()).hexdigest()[:8]
    )
    
    return plan

# Crear y mostrar el plan de vuelo
plan_vuelo = crear_plan_vuelo()
print("📋 Plan de Vuelo del Cohete:")
print("=" * 80)
display(plan_vuelo)

## 🌳 AST: Árbol de Sintaxis Abstracta

**¿Qué es un AST?**

Un AST (Abstract Syntax Tree) es como un árbol genealógico de decisiones. Imagina que cada decisión que toma el cohete es un nodo en un árbol:

- La **raíz** es el inicio del vuelo
- Cada **rama** representa una posible acción
- Cada **hoja** es un resultado final

**¿Por qué se llama "abstracta"?**

Porque no contiene todos los detalles de implementación, solo la estructura lógica de las decisiones. Es como un mapa conceptual que muestra las relaciones, no los pasos específicos.

**¿Cómo evitamos usar `if`?**

En lugar de escribir muchos `if-else`, usamos un diccionario que mapea estados a acciones. Es como tener un manual donde buscas tu situación actual y te dice qué hacer, sin necesidad de preguntar "¿es esto?", "¿es aquello?".

**Aplicación al control del cohete:**

El AST representa todas las posibles transiciones de estado del cohete. Cada nodo es un estado (fase) y cada conexión es una transición. El sistema consulta el árbol para saber qué hacer en cada momento.

In [ ]:
class SistemaControlAST:
    """Sistema de control basado en AST sin if explícitos"""
    
    def __init__(self, plan_vuelo):
        self.plan = plan_vuelo
        
        # Diccionario de transiciones (AST como estructura de datos)
        self.transiciones = {
            'despegue': {'siguiente': 'ascenso_inicial', 'accion': 'encender_motores'},
            'ascenso_inicial': {'siguiente': 'ascenso', 'accion': 'mantener_empuje'},
            'ascenso': {'siguiente': 'ascenso_medio', 'accion': 'ajustar_vectores'},
            'ascenso_medio': {'siguiente': 'ascenso_alto', 'accion': 'control_activo'},
            'ascenso_alto': {'siguiente': 'pre_orbital', 'accion': 'reducir_empuje'},
            'pre_orbital': {'siguiente': 'orbital', 'accion': 'preparar_orbital'},
            'orbital': {'siguiente': 'orbital_medio', 'accion': 'insercion_orbital'},
            'orbital_medio': {'siguiente': 'orbital_final', 'accion': 'estabilizar'},
            'orbital_final': {'siguiente': None, 'accion': 'orbita_establecida'}
        }
    
    def obtener_fase(self, altitud):
        """Busca la fase en la tabla hash por altitud"""
        idx = (self.plan['altitud'] - altitud).abs().idxmin()
        return self.plan.iloc[idx]
    
    def ejecutar_transicion(self, fase_actual):
        """Ejecuta transición sin if, usando diccionario"""
        return self.transiciones.get(fase_actual, {'siguiente': None, 'accion': 'mantener'})

# Crear sistema de control
sistema_ast = SistemaControlAST(plan_vuelo)
print("✅ Sistema de control AST inicializado")

## 📊 Visualización del AST

**¿Qué vamos a ver?**

Vamos a dibujar el árbol de decisiones del cohete. Cada caja representa una fase del vuelo, y las flechas muestran cómo pasa de una fase a otra. Es como ver el mapa completo del viaje del cohete desde la Tierra hasta el espacio.

Los colores ayudan a distinguir:
- **Azul claro**: Fases de ascenso
- **Verde**: Fase orbital final

In [ ]:
def visualizar_ast(sistema):
    """Crea visualización del AST con Graphviz"""
    
    dot = graphviz.Digraph(comment='AST del Sistema de Control')
    dot.attr(rankdir='TB')
    dot.attr('node', shape='box', style='filled', fillcolor='lightblue')
    
    # Agregar nodos y transiciones
    for fase, info in sistema.transiciones.items():
        dot.node(fase, f"{fase}\n{info['accion']}")
        
        if info['siguiente']:
            dot.edge(fase, info['siguiente'])
    
    # Marcar nodo final
    dot.node('orbital_final', 'orbital_final\norbita_establecida', fillcolor='lightgreen')
    
    return dot

# Visualizar el AST
ast_grafo = visualizar_ast(sistema_ast)
display(ast_grafo)

## 🎯 Filtro de Kalman

**¿Qué es el filtro de Kalman?**

Imagina que estás en una habitación oscura y 5 personas te dicen dónde está la puerta. Cada uno te da una dirección ligeramente diferente porque tienen información imperfecta (ruido). El filtro de Kalman es como un árbitro sabio que escucha a todos y calcula la respuesta más probable, dándole más peso a las opiniones más confiables.

**Analogía con testigos:**

- Testigo A dice: "El cohete está desviado 3.2°" (sensor con ruido)
- Testigo B dice: "El cohete está desviado 2.8°" (otro sensor)
- Testigo C dice: "El cohete está desviado 3.1°" (modelo predictivo)

El filtro de Kalman combina estas opiniones considerando:
1. Qué tan confiable es cada testigo (covarianza de medición)
2. Qué esperábamos basándonos en el pasado (predicción)

Resultado: "La desviación real es probablemente 3.0°"

**¿Por qué lo usamos en el cohete?**

Los sensores del cohete (giroscopios, acelerómetros) dan mediciones con ruido. El filtro de Kalman suaviza estas mediciones para tener una estimación más precisa de la posición y desviación real.

In [ ]:
class FiltroKalman:
    """Filtro de Kalman simplificado para suavizar desviaciones"""
    
    def __init__(self, q=0.01, r=0.1):
        """
        q: ruido del proceso (qué tan impredecible es el sistema)
        r: ruido de la medición (qué tan ruidosos son los sensores)
        """
        self.q = q  # Varianza del proceso
        self.r = r  # Varianza de la medición
        self.x = 0.0  # Estado estimado (desviación)
        self.p = 1.0  # Covarianza del error
    
    def actualizar(self, medicion):
        """Actualiza el filtro con una nueva medición"""
        
        # Predicción
        self.p = self.p + self.q
        
        # Ganancia de Kalman (qué tanto confiamos en la medición vs predicción)
        k = self.p / (self.p + self.r)
        
        # Actualización
        self.x = self.x + k * (medicion - self.x)
        self.p = (1 - k) * self.p
        
        return self.x

# Ejemplo de uso
kalman = FiltroKalman()
mediciones_ruidosas = [3.2, 2.8, 3.1, 2.9, 3.0]
print("🔍 Ejemplo de suavizado con Filtro de Kalman:")
print("=" * 50)
for i, medicion in enumerate(mediciones_ruidosas):
    estimacion = kalman.actualizar(medicion)
    print(f"Medición {i+1}: {medicion:.2f}° → Estimación suavizada: {estimacion:.2f}°")

## 🎛️ Controlador PID

**¿Qué es PID?**

PID significa:
- **P**roporcional: Corrección proporcional al error
- **I**ntegral: Corrección acumulada de errores pasados
- **D**erivativo: Corrección basada en tendencias

**Analogía de la ducha:**

Estás en la ducha ajustando la temperatura:

1. **Proporcional (P)**: Si el agua está MUY fría, mueves la perilla MUCHO hacia caliente. Si está poco fría, la mueves POCO. La corrección es proporcional al error.

2. **Integral (I)**: Si el agua ha estado fría por mucho tiempo (error acumulado), aumentas más el calor para compensar todo ese tiempo de sufrimiento.

3. **Derivativo (D)**: Si notas que el agua se está enfriando rápidamente (tendencia), mueves la perilla antes de que se ponga muy fría, anticipándote al problema.

**En nuestro caso:**

Usamos solo el componente **Proporcional (P)** para simplificar. Si el cohete está desviado 3°, aplicamos una corrección proporcional a esos 3°. Mientras mayor la desviación, mayor la corrección.

**Fórmula:** corrección = Kp × error

Donde Kp es una constante que determina qué tan agresiva es la corrección.

In [ ]:
class ControladorPID:
    """Controlador PID simplificado (solo componente P)"""
    
    def __init__(self, kp=0.5):
        """
        kp: Ganancia proporcional (qué tan agresiva es la corrección)
        """
        self.kp = kp
    
    def calcular(self, error):
        """Calcula la corrección proporcional al error"""
        correccion = self.kp * error
        return correccion

# Ejemplo de uso
pid = ControladorPID(kp=0.5)
errores = [3.0, 2.0, 1.0, 0.5, 0.1]
print("⚙️ Ejemplo de corrección PID:")
print("=" * 50)
for i, error in enumerate(errores):
    correccion = pid.calcular(error)
    print(f"Paso {i+1}: Error {error:.1f}° → Corrección {correccion:.2f}°")

## 🚀 Simulador de Vuelo

**¿Qué hace el simulador?**

El simulador es el corazón de nuestro notebook. Combina todos los componentes anteriores:

1. **Consulta el plan de vuelo** (tabla hash) para saber en qué fase está
2. **Usa el AST** para decidir qué acción tomar
3. **Aplica el filtro de Kalman** para suavizar mediciones ruidosas
4. **Usa el PID** para corregir desviaciones

Es como un piloto automático que:
- Lee el manual de vuelo (plan)
- Toma decisiones lógicas (AST)
- Filtra ruido de sensores (Kalman)
- Corrige el rumbo (PID)

Simularemos el vuelo paso a paso con pausas para que veas cómo evoluciona en tiempo real.

In [ ]:
class SimuladorVuelo:
    """Simulador completo del vuelo del cohete"""
    
    def __init__(self, sistema_ast, con_viento=False):
        self.sistema = sistema_ast
        self.con_viento = con_viento
        self.kalman = FiltroKalman(q=0.01, r=0.1)
        self.pid = ControladorPID(kp=0.5)
        
        # Estado del cohete
        self.altitud = 0
        self.desviacion = 0.0
        self.desviacion_real = 0.0
        
        # Historial para gráficas
        self.historial = {
            'altitud': [],
            'desviacion': [],
            'desviacion_filtrada': [],
            'fase': [],
            'accion': []
        }
    
    def simular_viento(self, altitud):
        """Simula perturbación de viento según altitud"""
        if not self.con_viento:
            return 0.0
        
        # Viento más fuerte en ciertas altitudes
        viento_map = {
            (0, 5000): 0.5,
            (5000, 20000): 2.0,
            (20000, 50000): 3.0,
            (50000, 100000): 1.5,
            (100000, 200000): 0.3
        }
        
        for (alt_min, alt_max), intensidad in viento_map.items():
            if alt_min <= altitud < alt_max:
                return np.random.normal(intensidad, 0.5)
        
        return 0.0
    
    def ejecutar_paso(self):
        """Ejecuta un paso de simulación"""
        
        # Obtener fase actual del plan de vuelo
        info_fase = self.sistema.obtener_fase(self.altitud)
        fase_actual = info_fase['fase']
        
        # Ejecutar transición usando AST
        transicion = self.sistema.ejecutar_transicion(fase_actual)
        accion = transicion['accion']
        
        # Simular viento (perturbación)
        perturbacion = self.simular_viento(self.altitud)
        self.desviacion_real += perturbacion
        
        # Agregar ruido a la medición
        medicion_ruidosa = self.desviacion_real + np.random.normal(0, 0.3)
        
        # Filtrar con Kalman
        desviacion_filtrada = self.kalman.actualizar(medicion_ruidosa)
        
        # Calcular corrección con PID
        correccion = self.pid.calcular(desviacion_filtrada)
        
        # Aplicar corrección
        self.desviacion_real -= correccion
        
        # Guardar en historial
        self.historial['altitud'].append(self.altitud)
        self.historial['desviacion'].append(medicion_ruidosa)
        self.historial['desviacion_filtrada'].append(desviacion_filtrada)
        self.historial['fase'].append(fase_actual)
        self.historial['accion'].append(accion)
        
        return fase_actual, accion, medicion_ruidosa, desviacion_filtrada, correccion
    
    def simular_vuelo_completo(self, pasos=50, delay=0.2):
        """Simula el vuelo completo con visualización en tiempo real"""
        
        altitud_max = 200000
        incremento = altitud_max / pasos
        
        escenario = "🌬️ CON VIENTO" if self.con_viento else "🟢 NOMINAL"
        print(f"\n{'='*80}")
        print(f"🚀 INICIANDO SIMULACIÓN - Escenario: {escenario}")
        print(f"{'='*80}\n")
        
        for paso in range(pasos):
            self.altitud = paso * incremento
            
            fase, accion, desv_medida, desv_filtrada, correccion = self.ejecutar_paso()
            
            # Log del paso
            print(f"Paso {paso+1:2d}/{pasos} | "
                  f"Alt: {self.altitud:>7.0f}m | "
                  f"Fase: {fase:>16s} | "
                  f"Desv: {desv_medida:>5.2f}° → {desv_filtrada:>5.2f}° | "
                  f"Correc: {correccion:>5.2f}° | "
                  f"Acción: {accion}")
            
            time.sleep(delay)
        
        print(f"\n{'='*80}")
        print(f"✅ ÓRBITA ESTABLECIDA - Simulación completada")
        print(f"{'='*80}\n")

print("✅ Simulador de vuelo listo")

## 🟢 Escenario 1: Vuelo Nominal (Sin Viento)

**Happy Path - Todo sale perfecto**

En este escenario ideal:
- No hay viento ni perturbaciones externas
- El cohete sigue su trayectoria planificada
- La desviación se mantiene cerca de cero
- Solo hay pequeño ruido de sensores que el Kalman suaviza

Observa cómo el sistema consulta el plan de vuelo, ejecuta las acciones del AST y mantiene el control perfecto.

In [ ]:
# Crear simulador para vuelo nominal
simulador_nominal = SimuladorVuelo(sistema_ast, con_viento=False)

# Ejecutar simulación
simulador_nominal.simular_vuelo_completo(pasos=50, delay=0.2)

## 📊 Gráficas del Escenario 1: Vuelo Nominal

**¿Qué muestran estas gráficas?**

1. **Desviación vs Altitud**: Vemos cómo la desviación se mantiene cerca de cero durante todo el vuelo. La línea azul (medición ruidosa) tiene pequeñas fluctuaciones por el ruido del sensor, pero la línea roja (filtrada por Kalman) es suave y estable.

2. **Trayectoria de Altitud**: Muestra el ascenso progresivo del cohete desde 0 hasta 200,000 metros (200 km), llegando a la órbita baja terrestre.

En un vuelo perfecto, esperamos líneas casi planas en la desviación y una curva ascendente limpia en la altitud.

In [ ]:
def graficar_resultados(simulador, titulo_escenario):
    """Genera gráficas de resultados de la simulación"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gráfica 1: Desviación vs Altitud
    ax1.plot(simulador.historial['altitud'], 
             simulador.historial['desviacion'], 
             'b-', alpha=0.3, label='Medición con ruido')
    ax1.plot(simulador.historial['altitud'], 
             simulador.historial['desviacion_filtrada'], 
             'r-', linewidth=2, label='Filtrado (Kalman)')
    ax1.set_xlabel('Altitud (m)', fontsize=12)
    ax1.set_ylabel('Desviación (°)', fontsize=12)
    ax1.set_title(f'Desviación vs Altitud - {titulo_escenario}', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Gráfica 2: Altitud por paso
    pasos = range(len(simulador.historial['altitud']))
    ax2.plot(pasos, simulador.historial['altitud'], 'g-', linewidth=2)
    ax2.set_xlabel('Paso de simulación', fontsize=12)
    ax2.set_ylabel('Altitud (m)', fontsize=12)
    ax2.set_title(f'Trayectoria de Altitud - {titulo_escenario}', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Graficar resultados del vuelo nominal
graficar_resultados(simulador_nominal, "Escenario Nominal 🟢")

## 🌬️ Escenario 2: Vuelo con Viento

**Escenario Realista - Enfrentando perturbaciones**

Ahora simulamos condiciones reales:
- **Viento variable** según la altitud (más fuerte en ciertas capas atmosféricas)
- El cohete se desvía hasta **3° en algunas fases**
- El **filtro de Kalman** suaviza las mediciones ruidosas de los sensores
- El **controlador PID** calcula correcciones para volver al rumbo

**¿Qué observaremos?**

Verás cómo:
1. El viento causa desviaciones reales
2. Los sensores miden con ruido (línea azul temblorosa)
3. Kalman filtra el ruido (línea roja suave)
4. El PID aplica correcciones proporcionales
5. El cohete regresa gradualmente al rumbo correcto

Es como ver a un piloto experto luchando contra turbulencias y manteniendo el control.

In [ ]:
# Crear simulador para vuelo con viento
simulador_viento = SimuladorVuelo(sistema_ast, con_viento=True)

# Ejecutar simulación
simulador_viento.simular_vuelo_completo(pasos=50, delay=0.2)

## 📊 Gráficas del Escenario 2: Vuelo con Viento

**¿Qué diferencias veremos con el escenario nominal?**

1. **Desviación vs Altitud**: 
   - La línea azul muestra mediciones mucho más ruidosas
   - Hay picos de desviación donde el viento es más fuerte (5-50 km de altitud)
   - La línea roja (Kalman) suaviza efectivamente el ruido
   - Se ve cómo el sistema corrige gradualmente, volviendo la desviación hacia cero

2. **Trayectoria de Altitud**: 
   - Similar al vuelo nominal porque el ascenso continúa a pesar del viento
   - El viento afecta la orientación, no la velocidad de ascenso

**Observa el poder del Kalman + PID trabajando juntos:**
- Kalman filtra el ruido del sensor → sabemos dónde estamos realmente
- PID calcula la corrección → sabemos qué hacer
- El cohete corrige su rumbo → volvemos al plan de vuelo

In [ ]:
# Graficar resultados del vuelo con viento
graficar_resultados(simulador_viento, "Escenario con Viento 🌬️")

## 📈 Comparación entre Escenarios

**¿Cuál es la diferencia entre ambos vuelos?**

Vamos a comparar directamente ambos escenarios en una sola gráfica para ver el impacto del viento y la efectividad del sistema de control.

**Lo que buscaremos:**
- En el vuelo nominal (verde), desviación casi nula
- En el vuelo con viento (rojo), desviaciones visibles pero controladas
- Ambos cohetes llegan exitosamente a órbita

Esto demuestra que nuestro sistema de control es **robusto**: funciona bien tanto en condiciones ideales como adversas.

In [ ]:
def comparar_escenarios(sim_nominal, sim_viento):
    """Compara ambos escenarios en una gráfica"""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Desviación filtrada de ambos escenarios
    ax.plot(sim_nominal.historial['altitud'], 
            sim_nominal.historial['desviacion_filtrada'], 
            'g-', linewidth=2.5, label='Nominal (sin viento)', alpha=0.8)
    
    ax.plot(sim_viento.historial['altitud'], 
            sim_viento.historial['desviacion_filtrada'], 
            'r-', linewidth=2.5, label='Con viento (corregido)', alpha=0.8)
    
    ax.set_xlabel('Altitud (m)', fontsize=13)
    ax.set_ylabel('Desviación Filtrada (°)', fontsize=13)
    ax.set_title('Comparación: Vuelo Nominal vs Vuelo con Viento', 
                 fontsize=15, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
    
    plt.tight_layout()
    plt.show()

# Comparar ambos escenarios
comparar_escenarios(simulador_nominal, simulador_viento)

## 📊 Estadísticas Finales

**Análisis cuantitativo del rendimiento**

Vamos a calcular métricas clave para evaluar qué tan bien funcionó el sistema de control en cada escenario:

- **Desviación máxima**: El peor momento de desviación
- **Desviación promedio**: Qué tan desviado estuvo en general
- **Desviación final**: Qué tan preciso llegó a órbita

Estas métricas nos dicen si el cohete fue exitoso en mantener su trayectoria.

In [ ]:
def calcular_estadisticas(simulador, nombre_escenario):
    """Calcula estadísticas de rendimiento"""
    
    desv_filtrada = np.array(simulador.historial['desviacion_filtrada'])
    
    stats = {
        'Escenario': nombre_escenario,
        'Desviación Máxima (°)': np.max(np.abs(desv_filtrada)),
        'Desviación Promedio (°)': np.mean(np.abs(desv_filtrada)),
        'Desviación Final (°)': np.abs(desv_filtrada[-1]),
        'Altitud Final (m)': simulador.historial['altitud'][-1],
        'Fases Completadas': len(set(simulador.historial['fase']))
    }
    
    return stats

# Calcular estadísticas de ambos escenarios
stats_nominal = calcular_estadisticas(simulador_nominal, "Nominal 🟢")
stats_viento = calcular_estadisticas(simulador_viento, "Con Viento 🌬️")

# Crear DataFrame comparativo
df_stats = pd.DataFrame([stats_nominal, stats_viento])

print("\n" + "="*80)
print("📊 ESTADÍSTICAS FINALES DE RENDIMIENTO")
print("="*80 + "\n")
display(df_stats)

print("\n" + "="*80)
print("✅ CONCLUSIÓN:")
print("="*80)
print("Ambos cohetes alcanzaron órbita exitosamente.")
print(f"El vuelo con viento tuvo {stats_viento['Desviación Máxima (°)']:.2f}° de desviación máxima,")
print(f"pero el sistema Kalman + PID logró corregirla efectivamente.")
print(f"Desviación final: {stats_viento['Desviación Final (°)']:.3f}° (excelente precisión)")
print("="*80 + "\n")

## 🎓 Resumen de Conceptos Implementados

**¿Qué aprendimos con este notebook?**

### 1. **Tabla Hash con Plan de Vuelo**
- Usamos pandas para crear una tabla de búsqueda rápida
- SHA1 genera identificadores únicos para cada altitud
- Permite consultar instantáneamente qué hacer en cada momento

### 2. **AST (Árbol de Sintaxis Abstracta)**
- Estructura de control sin `if` explícitos
- Usa diccionarios para mapear estados → acciones
- Visualizable con Graphviz para entender el flujo

### 3. **Filtro de Kalman**
- Combina mediciones ruidosas con predicciones
- Estima el estado real del cohete (desviación verdadera)
- Esencial cuando los sensores tienen ruido

### 4. **Controlador PID (componente P)**
- Calcula correcciones proporcionales al error
- Mientras mayor la desviación, mayor la corrección
- Devuelve el cohete gradualmente al rumbo correcto

### 5. **Integración de Sistemas**
- Plan de vuelo (qué hacer) + AST (cómo decidir) + Kalman (dónde estoy) + PID (cómo corregir)
- Trabajan juntos como un sistema de control robusto
- Funciona en condiciones ideales y adversas

---

**Aplicaciones reales:**

Este tipo de sistemas se usa en:
- 🚀 Cohetes reales (SpaceX Falcon 9, NASA SLS)
- ✈️ Pilotos automáticos de aviones
- 🚗 Vehículos autónomos
- 🤖 Robots móviles
- 🛰️ Control de satélites

Los principios son los mismos: medir, filtrar ruido, calcular error, corregir.

---

**Autor:** Omar Velázquez (https://www.linkedin.com/in/ovelazquezj)

**Licencia:** CC BY 4.0 (2025)